In [ ]:
!pip install "mlconfgen[full]==0.4.0"

## Download the weights from HuggingFace
> https://huggingface.co/Membrizard/ml_conformer_generator

`edm_moi_chembl_15_39.pt`

`edm_moi_chembl_6_39_fragments.pt` -> For IFM and generation of molecules from 6 to 39 heavy atoms

`adj_mat_seer_chembl_15_39.pt`

## 1. Basic Fine-Tuning

Fine-tune molecular generation for a specific task using parameters such as a spatial reference, fixed fragment, and variance.
The scoring_function can be any Python callable that accepts a list of RDKit Mol objects or None's and returns a list of float scores.

Fine-tuning saves `latest_checkpoint.pt` and `best_checkpoint.pt` in the specified directory. These checkpoints can be used either to initialize a fine-tuned generator or to load fine-tuned weights into an existing one.

Generators with fine-tuned checkpoints can be exported to ONNX. `MLConformerGeneratorONNX` also supports loading fine-tuned checkpoints, provided the checkpoint has been exported to ONNX format.

> **NOTE**
> If `scoring_function` is None, a default scoring function enforcing validity is applied for RL.


In [ ]:
import torch
from rdkit import Chem, RDLogger
from mlconfgen import MLConformerGenerator

RDLogger.DisableLog('rdApp.*')

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps:0")
else:
    device = torch.device("cpu")

print(f"Intitialising model on {device}")

generator = MLConformerGenerator(
                                 edm_weights="./edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="./adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=10,
                                )

ref_mol = Chem.MolFromMolFile('./assets/demo_files/ceyyag.mol')


generator.fine_tune(
                  scoring_function=None,
                  reference_conformer=ref_mol,
                  variance= 1,
                  n_epochs=20,
                  train_batch_size=16,
                  eval_batch_size=16,
                  lambda_edm_adapter=1.5,
                  lambda_edm_reg=0.01,
                  learning_rate= 8e-5,
                  sigma=60.0,
                  temperature=1.5,
                  n_samples_per_mol=8,
                  eval_every=2,
                  save_dir="./rl_checkpoints",
    
)



## 2. REINVENT4 compatible objective-guided Fine-Tuning

`MLConformerGenerator` can use REINVENT4 scoring as the objective for fine-tuning. To do this, initialize `ReinventScoreWrapper` with a REINVENT scoring configuration file. This requires REINVENT4 to be installed.

> ! git clone https://github.com/MolecularAI/REINVENT4.git --depth 1
> 
> ! cd REINVENT4 && python install.py <processor> && cd ..

In [ ]:
import torch

from rdkit import Chem, RDLogger
from mlconfgen import MLConformerGenerator

from mlconfgen.rl_fine_tuning.reinvent_score_wrapper import ReinventScoreWrapper

RDLogger.DisableLog('rdApp.*')

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps:0")
else:
    device = torch.device("cpu")


generator = MLConformerGenerator(
                                 edm_weights="./edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="./adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=10,
                                )
ref_mol = Chem.MolFromMolFile("./assets/demo_files/chembl63_p1.mol")

scoring_fn = ReinventScoreWrapper("./assets/demo_files/scoring_config.toml")


generator.fine_tune(
                  scoring_function=scoring_fn,
                  reference_conformer=ref_mol,
                  variance= 1,
                  n_epochs=20,
                  train_batch_size=128,
                  eval_batch_size=128,
                  lambda_edm_adapter=1.5,
                  lambda_edm_reg=0.02,
                  learning_rate= 8e-5,
                  sigma=128.0,
                  temperature=1.5,
                  n_samples_per_mol=16,
                  eval_every=5,
                  save_dir="./rl_checkpoints",
    
)